# AcademicForge AMD ROCm GPU VM Setup

This notebook guides you through setting up and running AcademicForge on your remote AMD GPU VM.

### Overview
1. **Clean Ports:** Free up port 8000 and 8501 from any zombie processes.
2. **ROCm PyTorch:** Install or verify the ROCm-enabled PyTorch build.
3. **Configure Credentials:** Set your HF Token and Fireworks Key.
4. **Start Servers:** Launch FastAPI and Streamlit using the virtualenv interpreter.
5. **Open App:** Use the notebook provider proxy for Streamlit on port 8501.


### Step 1: Clean Port 8000 & 8501
Run this cell to kill any zombie/orphan processes currently occupying port 8000 or 8501.

In [ ]:
import os
import signal

def kill_process_on_port(port):
    try:
        with open("/proc/net/tcp") as f:
            lines = f.readlines()
        hex_port = f"{port:04X}"
        inode = next((l.split()[9] for l in lines if l.split()[1].endswith(hex_port)), None)
        if inode:
            killed = False
            for pid in os.listdir("/proc"):
                if pid.isdigit():
                    try:
                        fd_dir = f"/proc/{pid}/fd"
                        for fd in os.listdir(fd_dir):
                            if f"socket:[{inode}]" in os.readlink(f"{fd_dir}/{fd}"):
                                os.kill(int(pid), 9)
                                print(f"✅ Successfully killed process {pid} on port {port}")
                                killed = True
                    except Exception:
                        pass
            if not killed:
                print(f"ℹ️ Port {port} has open sockets but process could not be matched.")
        else:
            print(f"✅ Port {port} is already free.")
    except FileNotFoundError:
        # Fallback if not on Linux
        print(f"ℹ️ Skipping port check (not on a Linux /proc system).")
    except Exception as e:
        print(f"⚠️ Error checking port {port}: {e}")

kill_process_on_port(8000)
kill_process_on_port(8501)

### Step 2: Install ROCm-enabled PyTorch (Optional)
> **Note:** PyTorch ROCm is already pre-installed inside the local virtual environment (`venv`). **You can skip this cell** unless you are doing a fresh installation from scratch.

In [ ]:
# Run this cell ONLY if you are setting up a fresh environment
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/rocm6.0
!pip install -r requirements-local.txt

### Step 3: Configure Credentials
Paste your Hugging Face and Fireworks API credentials here.


In [ ]:
# Set your credentials below
HF_TOKEN = "YOUR_HF_TOKEN_HERE"
FIREWORKS_API_KEY = "YOUR_FIREWORKS_API_KEY_HERE"


### Step 4: Start Backend and Frontend
This launches the FastAPI backend (on port 8000) and Streamlit frontend (on port 8501) using the `venv` interpreter.

In [ ]:
import subprocess
import time
import sys

# Set environment variables
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["FIREWORKS_API_KEY"] = FIREWORKS_API_KEY
os.environ["LOCAL_LLM_PROVIDER"] = "auto"
os.environ["LOCAL_LLM_MODEL"] = "google/gemma-2-2b-it"
os.environ["ACADEMICFORGE_BACKEND_URL"] = "http://127.0.0.1:8000"

# Use the venv Python binary to avoid ModuleNotFoundErrors
python_cmd = "venv/bin/python" if os.path.exists("venv") else sys.executable

print("Starting FastAPI backend on port 8000...")
backend = subprocess.Popen([python_cmd, "-m", "uvicorn", "backend.app:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(15)  # Wait for uvicorn to start and warm up the local model

print("Starting Streamlit frontend on port 8501...")
frontend = subprocess.Popen([
    python_cmd, "-m", "streamlit", "run", "frontend/streamlit_app.py",
    "--server.address", "0.0.0.0",
    "--server.port", "8501",
    "--server.baseUrlPath", "",
])
time.sleep(3)

print("\nServers are started in the background!")


### Step 5: Open Streamlit
Open Streamlit through your VM or notebook provider proxy on port 8501.


In [ ]:
INSTANCE_ID = "hf-308-940cf411"
print(f"Open: https://radeon-global.anruicloud.com/instances/{INSTANCE_ID}/proxy/8501/")
print("Backend health: https://radeon-global.anruicloud.com/instances/{}/proxy/8000/version".format(INSTANCE_ID))
